# Using Reasoning Models

So far in the examples, we have used a `gpt-4o` model. Models like `gpt-4o` and `gemini-1.5-flash` are great at following instructions, so you can have relatively detailed instructions in the selector prompt for the team and the system messages for each agent to guide their behavior.

However, if you are using a reasoning model like `o3-mini`, you will need to keep the "selector prompt" and" system messages" as simple and to the point as possible. This is because the reasoning models are already good at coming up with their own instructions given the `context` provided to them.

This also means that we don’t need a planning agent to break down the task anymore, since the `SelectorGroupChat` that uses a reasoning model can do that on its own.

In the following example, we will use o3-mini as the model for the agents and the team, and we will not use a planning agent. Also, we are keeping the selector prompt and system messages as simple as possible.

#### ***Reference URL***

https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/selector-group-chat.html#using-reasoning-models

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination

In [2]:
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
api_key= os.getenv('OPENAI_API_KEY')

In [3]:
# Model client

model_client= OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)
model_client

In [5]:
# Search Web Tool

def search_web_tool(query: str) -> str:
    # Simulate a web search
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [4]:
def percentage_change_tool(start:float, end:float) -> float:
    '''Calculate percentage change'''
    
    if start == 0:
        return 0
    return ((end - start) / start) * 100

In [6]:
# Termination Condition
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

text_mention_termination= TextMentionTermination(text="TERMINATE")
max_message_termination= MaxMessageTermination(max_messages=20)

combined_termination= text_mention_termination | max_message_termination

In [8]:
# model_client = OpenAIChatCompletionClient(model="o3-mini")

from autogen_agentchat.agents import UserProxyAgent

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""Use web search tool to find information.""",
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""Use tool to perform calculation. If you have not seen the data, ask for it.""",
)

user_proxy_agent = UserProxyAgent(
    "UserProxyAgent",
    description="A user to approve or disapprove tasks.",
)

selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
When the task is complete, let the user approve or disapprove the task.
"""

team = SelectorGroupChat(
    [web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=combined_termination,  # Use the same termination condition as before.
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
)


In [9]:
task= "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"
task

'Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?'

In [10]:
from autogen_agentchat.ui import Console

await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_ZqU296NDPKH2sLxmppWZOBPu', arguments='{"query": "Miami Heat player with the highest point in the 2006-2007 season and percentage change in his total rebounds between 2007-2008 and 2008-2009 seasons"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_ZqU296NDPKH2sLxmppWZOBPu', is_error=False)]
---------- ToolCallSummaryMessage (WebSearchAgent) ----------
Here are the total poi

TaskResult(messages=[TextMessage(id='e6015662-f527-48c4-81d6-78fa6ae9127e', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 38, 41, 353991, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), ToolCallRequestEvent(id='f193eb94-67e8-4b3a-9729-14627621dbaf', source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=259), metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 38, 52, 129345, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_ZqU296NDPKH2sLxmppWZOBPu', arguments='{"query": "Miami Heat player with the highest point in the 2006-2007 season and percentage change in his total rebounds between 2007-2008 and 2008-2009 seasons"}', name='search_web_tool')], type='ToolCallRequestEvent'), ToolCallExecutionEv

In [11]:
await team.reset()

In [12]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_0FMAmoBbxCtW1KVYwj2YB0jg', arguments='{"query": "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons? Miami Heat 2006-2007 season highest points, rebounds percentage change"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_0FMAmoBbxCtW1KVYwj2YB0jg',

TaskResult(messages=[TextMessage(id='2d10c4d1-c130-4bc6-bfe6-3c2fdc8626b6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 44, 22, 997880, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), ToolCallRequestEvent(id='134731bf-a786-4cdc-91de-1802b5fac693', source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=283), metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 44, 45, 327122, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_0FMAmoBbxCtW1KVYwj2YB0jg', arguments='{"query": "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons? Miami Heat 2006-2007 season highest points, rebo

In [14]:
# model_client = OpenAIChatCompletionClient(model="o3-mini")

from autogen_agentchat.agents import UserProxyAgent

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""Use web search tool to find information.""",
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""Use tool to perform calculation. If you have not seen the data, ask for it.""",
)

user_proxy_agent = UserProxyAgent(
    "UserProxyAgent",
    description="A user to approve or disapprove tasks.",
)

selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
When the task is complete, let the user approve or disapprove the task.
Once User approve then exit from the loop, If User seleact disapprove then goto the next agent.
"""

team = SelectorGroupChat(
    [web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=combined_termination,  # Use the same termination condition as before.
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
)


In [15]:
await team.reset()

In [16]:
task= "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"
task

'Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?'

In [17]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_5RM2MXeNu24NCfQC3p86xSB6', arguments='{"query": "Miami Heat player with highest points in the 2006-2007 season and percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_5RM2MXeNu24NCfQC3p86xSB6', is_error=False)]
---------- ToolCallSummaryMessage (WebSearchAgent) ----------
Here are the total po

TaskResult(messages=[TextMessage(id='5e2ef7e5-a2a0-416f-8a92-cf2a96b341b2', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 53, 35, 142432, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), ToolCallRequestEvent(id='b2c6bb74-d974-45ae-b3f7-398815be8108', source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=144), metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 53, 43, 891592, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_5RM2MXeNu24NCfQC3p86xSB6', arguments='{"query": "Miami Heat player with highest points in the 2006-2007 season and percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons"}', name='search_web_tool')], type='ToolCallRequestEvent'), ToolCallExecutionE

In [18]:
await team.reset()
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_z73ioixwzs7gYjltgZG3kzgv', arguments='{"query": "Dwyane Wade 2006-2007 Miami Heat highest points, and what was the percentage change in his total rebounds between 2007-2008 and 2008-2009 season?"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_z73ioixwzs7gYjltgZG3kzgv', is_error=False)]
---------- ToolCallSummaryMessage (WebSearchAgent) ----------
Here are the total points

TaskResult(messages=[TextMessage(id='1d7307c6-1450-48f2-8f3a-ed427e89d2e9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 59, 30, 469680, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), ToolCallRequestEvent(id='95dcba00-b2e5-4830-82a1-8b4229b0bf9d', source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=390), metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 59, 46, 989304, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_z73ioixwzs7gYjltgZG3kzgv', arguments='{"query": "Dwyane Wade 2006-2007 Miami Heat highest points, and what was the percentage change in his total rebounds between 2007-2008 and 2008-2009 season?"}', name='search_web_tool')], type='ToolCallRequestEvent'), ToolCallExecutionEvent

#### Structured Output

In [ ]:
TaskResult(
    messages=[
        TextMessage(id='1d7307c6-1450-48f2-8f3a-ed427e89d2e9', 
        source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 59, 30, 469680, tzinfo=datetime.timezone.utc), 
        content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), 
        
        ToolCallRequestEvent(id='95dcba00-b2e5-4830-82a1-8b4229b0bf9d', 
                             source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=103, completion_tokens=390), metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 59, 46, 989304, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_z73ioixwzs7gYjltgZG3kzgv', arguments='{"query": "Dwyane Wade 2006-2007 Miami Heat highest points, and what was the percentage change in his total rebounds between 2007-2008 and 2008-2009 season?"}', name='search_web_tool')], type='ToolCallRequestEvent'), 
                             
        ToolCallExecutionEvent(id='aea4f21c-55c3-4705-9444-a452c57c885f', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 59, 46, 991298, tzinfo=datetime.timezone.utc), 
                               content=[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_z73ioixwzs7gYjltgZG3kzgv', is_error=False)], type='ToolCallExecutionEvent'), 
                               
        ToolCallSummaryMessage(id='0c6ddfd4-ad33-44c9-bb80-a67db1423aac', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 10, 59, 46, 992300, tzinfo=datetime.timezone.utc), 
                               content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_z73ioixwzs7gYjltgZG3kzgv', arguments='{"query": "Dwyane Wade 2006-2007 Miami Heat highest points, and what was the percentage change in his total rebounds between 2007-2008 and 2008-2009 season?"}', name='search_web_tool')], results=[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_z73ioixwzs7gYjltgZG3kzgv', is_error=False)]), 
                               
        TextMessage(id='563a3536-4c3c-4c4c-8421-6e6831fbc2b6', 
                    source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=187, completion_tokens=537), metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 7, 160281, tzinfo=datetime.timezone.utc), 
                    content='I have identified that Dwyane Wade was the highest scoring Miami Heat player in the 2006-2007 season with 1,397 points. However, I need the total rebounds numbers for Dwyane Wade from the 2007-2008 and 2008-2009 seasons in order to calculate the percentage change. Could you please provide that data?', type='TextMessage'), 
                    
        ToolCallRequestEvent(id='606b69c3-ea3a-4867-8ca0-a1cd05eb753e', 
                             source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=304, completion_tokens=179), metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 22, 657907, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_0v5r07eI3kOiDuMLdroq3sL1', arguments='{"query": "Dwyane Wade total rebounds for 2007-2008 and 2008-2009 seasons rebounds"}', name='search_web_tool')], type='ToolCallRequestEvent'), 
                             
        ToolCallExecutionEvent(id='d0596552-6299-422d-ac06-8e833ff6a0af', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 22, 659906, tzinfo=datetime.timezone.utc), 
                               content=[FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214.', name='search_web_tool', call_id='call_0v5r07eI3kOiDuMLdroq3sL1', is_error=False)], type='ToolCallExecutionEvent'), 
                               
        ToolCallSummaryMessage(id='ca385cc7-f485-4b53-9115-157f75a16dc6', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 22, 660908, tzinfo=datetime.timezone.utc), 
                               content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214.', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_0v5r07eI3kOiDuMLdroq3sL1', arguments='{"query": "Dwyane Wade total rebounds for 2007-2008 and 2008-2009 seasons rebounds"}', name='search_web_tool')], results=[FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214.', name='search_web_tool', call_id='call_0v5r07eI3kOiDuMLdroq3sL1', is_error=False)]), 
                               
        ToolCallRequestEvent(id='85498b21-da41-4a04-8221-05b8b5755479', 
                             source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=373, completion_tokens=110), metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 32, 418901, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_YPbkmKOoK2wwL7LN0vjhHl43', arguments='{"query": "Dwyane Wade total rebounds for 2008-2009 season rebounds Miami Heat"}', name='search_web_tool')], type='ToolCallRequestEvent'), 
                             
        ToolCallExecutionEvent(id='fca933ee-d8c6-41be-824a-210a97643958', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 32, 418901, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398.', name='search_web_tool', call_id='call_YPbkmKOoK2wwL7LN0vjhHl43', is_error=False)], type='ToolCallExecutionEvent'), 
                               
        ToolCallSummaryMessage(id='d8b2ba2b-82c8-4a8a-a95d-0f8a9e25927d', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 32, 418901, tzinfo=datetime.timezone.utc), 
                               content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398.', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_YPbkmKOoK2wwL7LN0vjhHl43', arguments='{"query": "Dwyane Wade total rebounds for 2008-2009 season rebounds Miami Heat"}', name='search_web_tool')], results=[FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398.', name='search_web_tool', call_id='call_YPbkmKOoK2wwL7LN0vjhHl43', is_error=False)]), 
                               
        ToolCallRequestEvent(id='e0c6eeaf-18e8-4666-a7b2-ffdc05a60519', 
                             source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=330, completion_tokens=229), metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 49, 32222, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_9vNzcIo9U36pcQSa2LQplFlS', arguments='{"start": 214, "end": 398}', name='percentage_change_tool')], type='ToolCallRequestEvent'), 
                             
        ToolCallExecutionEvent(id='dd13e1bd-6666-410d-b9a7-3a9cbcd0cf71', 
                               source='DataAnalystAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 49, 32222, tzinfo=datetime.timezone.utc), 
                               content=[FunctionExecutionResult(content='85.98130841121495', name='percentage_change_tool', call_id='call_9vNzcIo9U36pcQSa2LQplFlS', is_error=False)], type='ToolCallExecutionEvent'), 
                               
        ToolCallSummaryMessage(id='9e6b7e25-266d-4cc6-a3c6-b6bbc10ea2ea', 
                               source='DataAnalystAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 0, 49, 32222, tzinfo=datetime.timezone.utc), 
                               content='85.98130841121495', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_9vNzcIo9U36pcQSa2LQplFlS', arguments='{"start": 214, "end": 398}', name='percentage_change_tool')], results=[FunctionExecutionResult(content='85.98130841121495', name='percentage_change_tool', call_id='call_9vNzcIo9U36pcQSa2LQplFlS', is_error=False)]), 
        
        TextMessage(id='04826d92-8c9f-40d0-a9ce-9378e71d4d06', 
                    source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=366, completion_tokens=146), metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 1, 4, 869416, tzinfo=datetime.timezone.utc), 
                    content='Dwyane Wade was the highest scoring Miami Heat player in the 2006‑2007 season with 1,397 points, and the percentage increase in his total rebounds from the 2007‑2008 season (214 rebounds) to the 2008‑2009 season (398 rebounds) was approximately 85.98%.', type='TextMessage'), 
                    
        UserInputRequestedEvent(id='0ad44291-b7bd-4edb-817d-70839d77a495', 
                                source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 1, 23, 311522, tzinfo=datetime.timezone.utc), request_id='a3657fa3-40e2-4d04-88ec-e2785312dded', 
                                content='', type='UserInputRequestedEvent'), 
                                
        TextMessage(id='42e065ce-3bce-4e34-9aed-79a7c6136dae', 
                    source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 11, 1, 27, 751750, tzinfo=datetime.timezone.utc), 
                    content='Approve. TERMINATE', type='TextMessage')], 

    stop_reason="Text 'TERMINATE' mentioned")